# Module 4 - RAG pipeline

## Step 1 : Data Prepration & EDA

In [2]:
from datasets import load_dataset
import pandas as pd 
import numpy as np
import re
import os
import joblib
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize
import torch

e:\ITI_AI\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Amod/mental_health_counseling_conversations")
df = pd.DataFrame(ds['train'])
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [4]:
print("Before cleaning:\n")
# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

Before cleaning:

Total Context & Response: 3512
Total null values: 
Context     0
Response    0
dtype: int64
Duplicates rows: 760
Duplicates Response: 1032
Duplicates Context: 2517


In [5]:
def clean_text(text):
    '''Clean text by removing URLs, HTML tags, extra spaces and newlines.'''
    text = str(text)

    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    # Remove quoted attribute values:  src="..."  or  src='...'  (with or without closing quote)
    text = re.sub(r'\w+\s*=\s*["\'][^"\']*["\']?', '', text)
    # Remove unquoted attribute values:  src=...
    text = re.sub(r'\w+\s*=\s*[^\s>"\']+', '', text)
    # Remove remaining tag shell  <...>  or  <...  (no closing >)
    text = re.sub(r'<[^>]*>?', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\b\S+\.(com|org|net|ca|io|pdf|html|htm)\S*', '', text)

    # Clean whitespace
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

df['Context'] = df['Context'].apply(clean_text)
df['Response'] = df['Response'].apply(clean_text)

# Delete rows with empty Context or Response, drop short rows and duplicates
df = df[(df['Context'] != "") & (df['Response'] != "")]
df = df[df['Context'].str.len() >= 15]
df = df[df['Response'].str.len() >= 15]
df = df.drop_duplicates()

In [6]:
print("After cleaning:\n")

# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

After cleaning:

Total Context & Response: 2021
Total null values: 
Context     0
Response    0
dtype: int64
Duplicates rows: 0
Duplicates Response: 1
Duplicates Context: 1191


In [7]:
# There are some 'Contexts' that have multiple 'Responses'.
dup_context = df.groupby('Context')['Response'].nunique().sort_values(ascending=False)
dup_context[dup_context > 1]

Context
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years. I’ve never had counseling about any of this. Do I have too many issues to address in counseling?                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [8]:
conflicts = (
    df.groupby('Context')
      .filter(lambda x: x['Response'].nunique() > 1)
      .sort_values('Context')
)

with open("conflicts_output.txt", "w", encoding="utf-8") as f:
    for context, group in conflicts.groupby('Context'):
        f.write("=" * 80 + "\n")
        f.write("CONTEXT:\n\n")
        f.write(str(context) + "\n")

        f.write("\nRESPONSES:\n\n")
        for i, response in enumerate(group['Response'].unique(), 1):
            f.write(f"{i}. {response}\n\n")

print("Saved to conflicts_output.txt")

Saved to conflicts_output.txt


In [9]:
# saving the cleaned dataframe to a csv file in data folder
df.to_csv('data/df_cleaned.csv', index=False)

### Handle mutiple Response of the same Context

In [10]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Precompute embeddings for all unique responses and cache to disk
emb_cache_path = os.path.join('data', 'response_embeddings.joblib')
responses = df['Response'].unique().tolist()

if os.path.exists(emb_cache_path):
    emb_lookup = joblib.load(emb_cache_path)
    print(f'Loaded cached embeddings for {len(emb_lookup)} responses')
else:
    print(f'Computing embeddings for {len(responses)} unique responses...')
    emb_array = model.encode(
        responses,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )
    emb_lookup = {r: emb_array[i] for i, r in enumerate(responses)}
    os.makedirs(os.path.dirname(emb_cache_path), exist_ok=True)
    joblib.dump(emb_lookup, emb_cache_path)
    print(f'Saved embeddings cache to {emb_cache_path}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2867.93it/s]


Loaded cached embeddings for 2022 responses


#### Clustering-based Summarization

In [11]:
def semantic_clustering_summarize(responses, emb_lookup, similarity_threshold=0.75):
    """ Cluster similar responses and return a representative response for each cluster.
    Args:
        responses (list): List of responses to cluster.
        emb_lookup (dict): Mapping response -> embedding (numpy array).
        similarity_threshold (float): Threshold for cosine similarity to consider responses as similar.
    Returns:
        list: List of representative responses for each cluster.
    """
    # if there is only one response
    if len(responses) <= 1:
        return responses

    # Build embeddings matrix from lookup; fall back to encoding missing responses
    missing = [r for r in responses if r not in emb_lookup]
    if missing:
        # encode missing ones on-the-fly
        missing_embs = model.encode(missing, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        for i, r in enumerate(missing):
            emb_lookup[r] = missing_embs[i]

    embeddings = np.vstack([emb_lookup[r] for r in responses])

    # Clustering
    clustering_model = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=1 - similarity_threshold)
    labels = clustering_model.fit_predict(embeddings)

    # Pick representative response
    unique_views = []

    for cluster_id in np.unique(labels):
        cluster_indices = np.where(labels == cluster_id)[0]
        cluster_embeddings = embeddings[cluster_indices]

        # centroid of cluster
        centroid = cluster_embeddings.mean(axis=0)
        # normalize centroid to unit length (avoid zero division)
        norm = np.linalg.norm(centroid)
        if norm > 0:
            centroid = centroid / (norm + 1e-12)

        # similarity to centroid
        similarities = cosine_similarity(cluster_embeddings, centroid.reshape(1, -1)).flatten()

        # best representative
        best_local_idx = np.argmax(similarities)
        best_idx = cluster_indices[best_local_idx]
        unique_views.append(responses[best_idx])

    return unique_views

In [12]:
final_data = []

# Process each Context (with progress bar)
for context, group in tqdm(df.groupby('Context'), total=df['Context'].nunique()):
    raw_responses = group['Response'].unique().tolist()
    summarized_views = semantic_clustering_summarize(raw_responses, emb_lookup, similarity_threshold=0.75)

    # Guard against empty summarization
    if not summarized_views:
        summarized_views = raw_responses[:1]

    # Format responses
    formatted_responses = (
        "\n".join([f"• {r}" for r in summarized_views])
        if len(summarized_views) > 1
        else summarized_views[0])

    final_data.append({'Context': context, 'Responses': formatted_responses})

# Save final dataframe
summarized_df = pd.DataFrame(final_data)
out_path = os.path.join('data', 'semantic_clustered_rag.csv')
summarized_df.to_csv(out_path, index=False)
print(f'Saved summarized dataframe to {out_path}')
summarized_df.head()

100%|██████████| 830/830 [00:08<00:00, 95.47it/s] 


Saved summarized dataframe to data\semantic_clustered_rag.csv


,Context,Responses
0,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your f...
1,A few years ago I was making love to my wife w...,• First step always is to do a medical rule ou...
2,A friend of mine taking psychology advised I g...,I admire your courage for stating your view ab...
3,A girl and I were madly in love. We dated for ...,"Hi Boise, I'm sorry that you've lost this love..."
4,"A lot of times, I avoid situations where I am ...",• Why not accept and tolerate that you natural...


## Step 2 : Chunking

In [13]:
# Chunking Config 
CHUNK_SIZE_CHARS = 1500   # ~375 tokens per chunk
OVERLAP_CHARS    = 150    # chars carried over between chunks
BATCH_SIZE       = 64
MODEL_NAME       = 'all-MiniLM-L6-v2'

OUT_CHUNKS = os.path.join('data', 'chunks.parquet')
OUT_EMB    = os.path.join('artifacts', 'chunk_embeddings.joblib')
OUT_META   = os.path.join('artifacts', 'index_metadata.joblib')

print('Chunking config ready.')
print(f'  Chunk size : {CHUNK_SIZE_CHARS} chars')
print(f'  Overlap    : {OVERLAP_CHARS} chars')
print(f'  Model      : {MODEL_NAME}')

Chunking config ready.
  Chunk size : 1500 chars
  Overlap    : 150 chars
  Model      : all-MiniLM-L6-v2


In [14]:
def split_bullets(text):
    """
    Split a Responses field into individual therapist answers.
    Handles bullet markers: •  -  *  1.  and newlines.
    """
    if not text:
        return []
    parts = re.split(r'\n+', str(text))
    bullets = []
    for p in parts:
        s = p.strip()
        if not s:
            continue
        s = re.sub(r'^\s*(?:•|\-|\*|\d+\.)\s*', '', s)
        s = s.strip()
        if s:
            bullets.append(s)
    return bullets


def chunk_text(text, size=1500, overlap=150):
    text = str(text).strip()
    if not text:
        return []
    if len(text) <= size:
        return [text]
    
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        # Hard cap: if single sentence exceeds size, split it
        if len(sentence) > size:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            overlap_text = sentence[-overlap:] if overlap > 0 else ""
            current_chunk = overlap_text
            for i in range(0, len(sentence), size):
                chunks.append(sentence[i:i+size])
            current_chunk = ""
        elif len(current_chunk) + len(sentence) + 1 > size:
            chunks.append(current_chunk.strip())
            overlap_text = current_chunk[-overlap:] if overlap > 0 else ""
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += " " + sentence
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks


In [15]:
# Load the semantic-clustered output from Step 1
chunking_df = pd.read_csv(os.path.join('data', 'semantic_clustered_rag.csv'))
print(f'Loaded {len(chunking_df):,} rows')
print(f'Columns: {chunking_df.columns.tolist()}')
chunking_df.head(3)

Loaded 830 rows
Columns: ['Context', 'Responses']


,Context,Responses
0,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your f...
1,A few years ago I was making love to my wife w...,• First step always is to do a medical rule ou...
2,A friend of mine taking psychology advised I g...,I admire your courage for stating your view ab...


In [16]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [17]:
records = []

for i, row in tqdm(chunking_df.iterrows(), total=len(chunking_df), desc='Chunking rows'):
    question = str(row.get('Context', row.iloc[0])).strip()

    # Support both 'Responses' and 'Response' column names
    responses_field = row.get('Responses') if 'Responses' in chunking_df.columns else row.get('Response')

    bullets = split_bullets(responses_field) if responses_field else []
    if not bullets:
        bullets = [responses_field] if responses_field else [question]

    for bi, bullet in enumerate(bullets):
        qa_text = f'Q: {question}\nA: {bullet}'
        for ci, chunk in enumerate(chunk_text(qa_text, size=CHUNK_SIZE_CHARS, overlap=OVERLAP_CHARS)):
            records.append({
                'context_id'       : int(i),
                'bullet_index'     : int(bi),
                'chunk_index'      : int(ci),
                'original_response': bullet,
                'text'             : chunk,
            })

chunks_df = pd.DataFrame(records)
print(f'Original rows : {len(chunking_df):,}')
print(f'Total chunks  : {len(chunks_df):,}')
chunks_df.head(3)

Chunking rows: 100%|██████████| 830/830 [00:00<00:00, 1272.20it/s]

Original rows : 830
Total chunks  : 2,380


,context_id,bullet_index,chunk_index,original_response,text
0,0,0,0,Hey! It takes a lot of courage to share your f...,Q: A few nights ago I talked to this girl I kn...
1,0,0,1,Hey! It takes a lot of courage to share your f...,our wallet - and when you need a self-esteem b...
2,1,0,0,First step always is to do a medical rule out ...,Q: A few years ago I was making love to my wif...


In [18]:
chunks_df['char_len'] = chunks_df['text'].str.len()

print('Chunk character length stats:')
print(chunks_df['char_len'].describe().round(1))
print()
print(f'Chunks under 500 chars  : {(chunks_df["char_len"] < 500).sum():,}')
print(f'Chunks 500–1500 chars   : {((chunks_df["char_len"] >= 500) & (chunks_df["char_len"] <= 1500)).sum():,}')
print(f'Chunks over 1500 chars  : {(chunks_df["char_len"] > 1500).sum():,}')

Chunk character length stats:
count    2380.0
mean      991.5
std       376.4
min       125.0
25%       679.8
50%      1024.0
75%      1363.0
max      1585.0
Name: char_len, dtype: float64

Chunks under 500 chars  : 306
Chunks 500–1500 chars   : 2,072
Chunks over 1500 chars  : 2


In [19]:
# Save chunks to Parquet for efficient storage and later embedding
os.makedirs('data', exist_ok=True)
chunks_df.to_parquet(OUT_CHUNKS, index=False)
print(f'Saved {len(chunks_df):,} chunks → {OUT_CHUNKS}')

Saved 2,380 chunks → data\chunks.parquet


In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading embedding model: {MODEL_NAME} on {device}...')
embed_model = SentenceTransformer(MODEL_NAME, device=device)

texts    = chunks_df['text'].tolist()
emb_list = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding batches'):
    batch = texts[i : i + BATCH_SIZE]
    emb   = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb_list.append(emb)

embeddings = np.vstack(emb_list)
print(f'Embeddings shape: {embeddings.shape}')  # (n_chunks, 384)

Loading embedding model: all-MiniLM-L6-v2 on cpu...


Embedding batches: 100%|██████████| 38/38 [02:06<00:00,  3.33s/it]

Embeddings shape: (2380, 384)


In [21]:
joblib.dump(embeddings,  OUT_EMB)
joblib.dump(chunks_df.to_dict(orient='records'), OUT_META)

print(f'Saved embeddings → {OUT_EMB}   shape={embeddings.shape}')
print(f'Saved metadata   → {OUT_META}  records={len(chunks_df):,}')

Saved embeddings → artifacts\chunk_embeddings.joblib   shape=(2380, 384)
Saved metadata   → artifacts\index_metadata.joblib  records=2,380


In [22]:
# Quick check: retrieve nearest chunk to a sample query
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

sample_query = 'I feel hopeless and do not know what to do'
q_emb = embed_model.encode([sample_query], normalize_embeddings=True)
scores = cos_sim(q_emb, embeddings)[0]
top_idx = scores.argsort()[::-1][:3]

print(f'Query: "{sample_query}"\n')
for rank, idx in enumerate(top_idx, 1):
    print(f'--- Rank {rank}  (score={scores[idx]:.4f}) ---')
    print(chunks_df.iloc[idx]['text'][:300])
    print()

Query: "I feel hopeless and do not know what to do"

--- Rank 1  (score=0.5746) ---
n to recognize that you are unhappy and unfulfilled in life. It sounds like you may be at that crossroads right now. Take one small step at at a time. Identify the worst offenders in your life that suck time and energy, and limit your contact and/or set some strong boundaries with those people so yo

--- Rank 2  (score=0.5451) ---
Q: I think about death all the time because I feel so alone. I want someone to love and someone to love me.
A: Feeling alone and/or isolated is almost always associated with being depressed. As humans, we need connection and interaction with others in order to feel satisfied. Given that you are freq

--- Rank 3  (score=0.5289) ---
Q: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never ge

## Step 3 : Store in VectorDB

In [23]:
from qdrant_client import QdrantClient
from dotenv import load_dotenv
load_dotenv()

# Initialize Qdrant client using environment variables for URL and API key
qdrant_client = QdrantClient(
    url     = os.getenv("QDRANT_URL"),
    api_key = os.getenv("QDRANT_API_KEY"),
)

In [24]:
# Recreate collection with appropriate vector size and distance metric
from qdrant_client.http.models import Distance, VectorParams
qdrant_client.recreate_collection(
    collection_name="mental_health_chunks",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,
    ),
)

C:\Users\moham\AppData\Local\Temp\ipykernel_18692\2192556220.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

In [25]:
from qdrant_client.http.models import PointStruct
import time

# Load your saved embeddings and chunks
embeddings = joblib.load('artifacts/chunk_embeddings.joblib')
metadata   = joblib.load('artifacts/index_metadata.joblib')

# Prepare points for Qdrant upload
points = [
    PointStruct(
        id=idx,
        vector=embeddings[idx].astype('float32').tolist(),
        payload={
            "context_id"        : rec["context_id"],
            "bullet_index"      : rec["bullet_index"],
            "chunk_index"       : rec["chunk_index"],
            "original_response" : rec["original_response"],
            "text"              : rec["text"],
        }
    )
    for idx, rec in enumerate(metadata)
]

# upload points in batches with retry logic 
# (if a batch fails, it will retry up to 3 times with a delay)

BATCH = 32
for i in tqdm(range(0, len(points), BATCH), desc='Uploading to Qdrant'):
    batch = points[i : i + BATCH]
    
    for attempt in range(3):
        try:
            qdrant_client.upsert(
                collection_name = "mental_health_chunks", 
                points = batch,
                wait = False)
            print(f"Uploaded {min(i+BATCH, len(points))}/{len(points)}")
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2)

print("All points uploaded")

Uploading to Qdrant:   1%|▏         | 1/75 [00:04<05:17,  4.30s/it]

Uploaded 32/2380


Uploading to Qdrant:   3%|▎         | 2/75 [00:08<04:54,  4.03s/it]

Uploaded 64/2380


Uploading to Qdrant:   4%|▍         | 3/75 [00:12<04:45,  3.96s/it]

...

Uploaded 2336/2380


Uploading to Qdrant:  99%|█████████▊| 74/75 [06:17<00:04,  4.50s/it]

Uploaded 2368/2380


Uploading to Qdrant: 100%|██████████| 75/75 [06:20<00:00,  5.08s/it]

Uploaded 2380/2380
All points uploaded


In [26]:
# Verify count in Qdrant matches number of points uploaded
info = qdrant_client.get_collection("mental_health_chunks")
print(f'Vectors in Qdrant : {info.points_count:,}')
print(f'Expected          : {len(points):,}')
assert info.points_count == len(points), 'Mismatch! Some points may not have uploaded.'

Vectors in Qdrant : 2,380
Expected          : 2,380


In [27]:
# ------------------------------------------------------------------
# Quick check: retrieve nearest chunk to a query using Qdrant search
# ------------------------------------------------------------------

# Reuse embed_model if already loaded, otherwise reload
if 'embed_model' not in dir():
    embed_model = SentenceTransformer(MODEL_NAME)

sample_query = 'I feel hopeless and do not know what to do'

q_emb = embed_model.encode(
    [sample_query],
    normalize_embeddings=True
)[0].tolist()

results = qdrant_client.query_points(
    collection_name="mental_health_chunks",
    query=q_emb,
    limit=3,
    with_payload=True,
)

print(f'Query: "{sample_query}"\n')

for rank, r in enumerate(results.points, 1):
    print(f'--- Rank {rank} (score={r.score:.4f}) ---')
    print(r.payload['text'][:300])
    print()

Query: "I feel hopeless and do not know what to do"

--- Rank 1 (score=0.5746) ---
n to recognize that you are unhappy and unfulfilled in life. It sounds like you may be at that crossroads right now. Take one small step at at a time. Identify the worst offenders in your life that suck time and energy, and limit your contact and/or set some strong boundaries with those people so yo

--- Rank 2 (score=0.5451) ---
Q: I think about death all the time because I feel so alone. I want someone to love and someone to love me.
A: Feeling alone and/or isolated is almost always associated with being depressed. As humans, we need connection and interaction with others in order to feel satisfied. Given that you are freq

--- Rank 3 (score=0.5289) ---
Q: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get a

In [28]:
from rank_bm25 import BM25Okapi

# Load chunk texts (already saved in Step 2)
# Position in this list == Qdrant point id  (because you uploaded with id=idx)
bm25_texts       = pd.read_parquet(os.path.join('data', 'chunks.parquet'))['text'].tolist()
tokenized_corpus = [text.lower().split() for text in bm25_texts]
bm25_index       = BM25Okapi(tokenized_corpus)

print(f"BM25 index built on {len(bm25_texts):,} documents")

BM25 index built on 2,380 documents


## Step 4 : Hybrid Retrieval (Semantic 0.7 + BM25 0.3)

In [29]:
SEMANTIC_WEIGHT = 0.7
BM25_WEIGHT     = 0.3
CANDIDATE_POOL  = 50   # how many candidates to pull from each method before re-ranking


def _normalize(score_dict):
    """Min-max normalize a {id: score} dict to [0, 1]."""
    if not score_dict:
        return {}
    values = list(score_dict.values())
    lo, hi = min(values), max(values)
    if hi == lo:
        return {k: 1.0 for k in score_dict}
    return {k: (v - lo) / (hi - lo) for k, v in score_dict.items()}


def retrieve_chunks_hybrid(query_text, top_k=5):
    """
    Hybrid retrieval: 0.7 * semantic + 0.3 * BM25

    1. Semantic  → ask Qdrant for top CANDIDATE_POOL results + scores
    2. BM25      → score all docs locally, take top CANDIDATE_POOL
    3. Union     → merge both candidate sets
    4. Normalize → scale each score set to [0, 1] independently
                   (needed because cosine scores ~0.4-0.9 and BM25 scores ~0-30)
    5. Combine   → final = 0.7 * sem_norm + 0.3 * bm25_norm
    6. Return    → fetch payloads from Qdrant for the top_k winners
    """

    # ── 1. Semantic search ──────────────────────────────────────────────────
    q_emb = embed_model.encode([query_text], normalize_embeddings=True)[0].tolist()

    semantic_hits = qdrant_client.query_points(
        collection_name="mental_health_chunks",
        query=q_emb,
        limit=CANDIDATE_POOL,
        with_payload=True,
    )
    semantic_scores = {r.id: r.score for r in semantic_hits.points}

    # ── 2. BM25 search ──────────────────────────────────────────────────────
    tokenized_query = query_text.lower().split()
    bm25_all_scores = bm25_index.get_scores(tokenized_query)  # array of length n_chunks

    bm25_top_ids = np.argsort(bm25_all_scores)[::-1][:CANDIDATE_POOL]
    bm25_scores  = {int(i): float(bm25_all_scores[i]) for i in bm25_top_ids}

    # ── 3. Union of both candidate sets ────────────────────────────────────
    all_ids = set(semantic_scores.keys()) | set(bm25_scores.keys())

    # ── 4. Normalize ────────────────────────────────────────────────────────
    sem_norm  = _normalize(semantic_scores)
    bm25_norm = _normalize(bm25_scores)

    # ── 5. Combine ──────────────────────────────────────────────────────────
    combined = []
    for cid in all_ids:
        sem   = sem_norm.get(cid,  0.0)
        bm25  = bm25_norm.get(cid, 0.0)
        score = SEMANTIC_WEIGHT * sem + BM25_WEIGHT * bm25
        combined.append((cid, score))

    combined.sort(key=lambda x: x[1], reverse=True)
    top_ids = [cid for cid, _ in combined[:top_k]]

    # ── 6. Fetch payloads from Qdrant ───────────────────────────────────────
    fetched       = qdrant_client.retrieve(
        collection_name="mental_health_chunks",
        ids=top_ids,
        with_payload=True,
    )
    id_to_payload = {r.id: r.payload for r in fetched}

    # Return in ranked order, preserving the combined score ranking
    return [id_to_payload[cid]["text"] for cid in top_ids if cid in id_to_payload]

### Testing Retrieval

In [30]:
query   = "I feel happy and do not know what to do"
results = retrieve_chunks_hybrid(query, top_k=3)

print(f'Query: "{query}"\n')
for i, text in enumerate(results, 1):
    print(f"--- Rank {i} ---")
    print(text[:300])
    print()

Query: "I feel happy and do not know what to do"

--- Rank 1 ---
Q: How do I make myself happy without the people who made me happy? Now that they’re gone, I feel sad. It’s been two months now but I seem to be unable to stay okay and independent.
A: It sounds like you have been feeling pretty down, since the loss of a relationship, and you're wondering how to be 

--- Rank 2 ---
l help you look at the barriers to happiness in your specific case and suggest a course of treatment. You are not alone, and you don't have to suffer. Keep asking questions and you will find your answers!

--- Rank 3 ---
n to recognize that you are unhappy and unfulfilled in life. It sounds like you may be at that crossroads right now. Take one small step at at a time. Identify the worst offenders in your life that suck time and energy, and limit your contact and/or set some strong boundaries with those people so yo



## Step 5 : Response

In [31]:
from openai import OpenAI

client_llm = OpenAI(
    api_key  = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPENAI_BASE_URL"),
)

MAIN_MODEL        = "llama-3.1-8b-instant"
TRANSLATOR_MODEL  = "llama-3.1-8b-instant"

### Language Detector Model

In [32]:
# Map your language detector's 3-letter output codes to full language names
# The LLM understands "Arabic" much better than "ara" in the prompt
LANGUAGE_NAMES = {
    "pt": "Portuguese",
    "bg": "Bulgarian",
    "zh": "Chinese",
    "th": "Thai",
    "ru": "Russian",
    "pl": "Polish",
    "ur": "Urdu",
    "sw": "Swahili",
    "tr": "Turkish",
    "es": "Spanish",
    "ar": "Arabic",
    "it": "Italian",
    "hi": "Hindi",
    "de": "German",
    "el": "Greek",
    "nl": "Dutch",
    "fr": "French",
    "vi": "Vietnamese",
    "en": "English",
    "ja": "Japanese"
}

svc_model = joblib.load("models/language_svc_model.pkl")
vectorizer = joblib.load("models/tfidf_vectorizer.pkl")

# create a function to predict the language of a given text
def predict_language(text):
    text_vec = vectorizer.transform([text])
    prediction = svc_model.predict(text_vec)
    return LANGUAGE_NAMES[prediction[0]]

In [33]:
def translate_to_english(text, source_language):
    """
    source_language is now a full name e.g. 'arabic', 'french'
    """
    response = client_llm.chat.completions.create(
        model    = TRANSLATOR_MODEL,
        messages = [
            {
                "role"   : "system",
                "content": (
                    f"You are a translation engine. "
                    f"Translate the following {source_language} text to English. "
                    f"Return ONLY the translated text — no explanation, "
                    f"no preamble, nothing else."
                )
            },
            {
                "role"   : "user",
                "content": text
            }
        ],
        max_tokens  = 512,
        temperature = 0.0,
    )
    return response.choices[0].message.content.strip()

### Emotional Model

In [34]:
EMOTION_TONE = {
    "sadness" : "Be empathetic and gentle. Acknowledge their pain and show you understand before offering help.",
    "joy"     : "Be warm and encouraging. Celebrate their positive feelings while offering supportive guidance.",
    "fear"    : "Be calm and reassuring. Help ground them and reduce their anxiety with clear, steady guidance.",
    "anger"   : "Be calm and non-confrontational. Validate their frustration without escalating, and guide them gently.",
    "love"    : "Be warm and supportive. Acknowledge their feelings of connection and guide them with care.",
    "surprise": "Be clear and informative. Help them process the unexpected situation with steady, factual support.",
}

In [35]:
def build_prompt(query, chunks, emotion, language):

    tone = EMOTION_TONE.get(emotion, "Be empathetic and supportive.")

    system_message = f"""You are a compassionate mental health support assistant.
Your role is to provide empathetic, grounded, and helpful responses based ONLY on the context provided.

Tone instruction: {tone}

Rules:
- Answer ONLY from the provided context. Do not invent information.
- If the context does not contain a relevant answer, say so honestly and suggest seeking professional help.
- Keep your response concise, warm, and easy to understand.
- You MUST respond in {language}.
"""

    context_block = "\n\n".join(
        [f"[Context {i+1}]:\n{chunk}" for i, chunk in enumerate(chunks)]
    )

    user_message = f"""Context from knowledge base:
{context_block}

User question: {query}
"""
    return system_message, user_message

In [67]:
def call_llm(system_message, user_message, history=[]):
    
    messages = [{"role": "system", "content": system_message}]
    
    # inject previous conversation turns
    messages.extend(history)
    
    # add the new user message
    messages.append({"role": "user", "content": user_message})

    response = client_llm.chat.completions.create(
        model       = MAIN_MODEL,
        messages    = messages,
        max_tokens  = 1024,
        temperature = 0.7,
    )
    return response.choices[0].message.content.strip()

In [68]:
def get_rag_answer(original_query, detected_language, emotion, history=[], top_k=5):

    if detected_language != "english":
        english_text = translate_to_english(original_query, detected_language)
    else:
        english_text = original_query

    chunks               = retrieve_chunks_hybrid(english_text, top_k=top_k)
    system_msg, user_msg = build_prompt(english_text, chunks, emotion, detected_language)
    answer               = call_llm(system_msg, user_msg, history)  # ← pass history

    return answer

In [41]:
query = "انا مبسوط جدا و الدنيا زى الفل تقدر تقزلى اعمل اية عشان احافظ على طاقتى الايجابية دى "
answer = get_rag_answer(
    original_query    = query,
    detected_language = predict_language(query),
    top_k             = 5
)

print(answer)

أنا سعيد جدًا بسماعك أنك تشعر بالسعادة والتفاؤل. هذا شيء رائع! تشعرك بالسعادة هي دليل على أنك قوي ويمكنك تحقيق الأشياء التي تريدها. استمر في العمل على توجيه هذا الطاقة الإيجابية نحو العالم. يبدو أنك تشعر بالراحة مع العالم ومستعدًا لتحقيق الأشياء الجيدة. هذا شيء يجب أن نستمتع به. استمر في توجيه هذا الطاقة الإيجابية نحو العالم، وستكون قادرًا على تحقيق الكثير من الأشياء الجيدة.


## Intent Classifier

In [ ]:
import os
import json
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_groq import ChatGroq   
import logging
import sys

load_dotenv()

True

In [44]:
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s [%(levelname)s] %(message)s",
    datefmt  = "%H:%M:%S",
    handlers = [logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

In [45]:
SYSTEM_PROMPT = """You are an advanced triage and routing assistant for a specialized mental health support system. 
Your sole responsibility is to analyze the user's input and categorize it into exactly ONE of the allowed intent classes.

CRITICAL INSTRUCTIONS:
1. Base your classification entirely on the core intent of the user's statement.
2. Rely heavily on the few-shot examples provided below to understand the classification boundaries.
3. Choose exactly one of the allowed categories. Do not invent new categories.

ALLOWED CATEGORIES AND DEFINITIONS:

- greeting: 
  The user is starting a conversation, saying hello, or checking if someone is online.
  (e.g., "Hi", "Hello", "Is anyone there?", "Good morning")

- goodbye: 
  The user is ending the conversation, signing off, or indicating they are leaving.
  (e.g., "Bye", "See you later", "Talk to you tomorrow", "I'm heading out")

- gratitude: 
  The user is thanking the system, expressing appreciation, or confirming that their issue was resolved.
  (e.g., "Thank you so much", "Thanks for the help", "That makes sense, thank you", "Appreciate it")

- asking_mental_health_question: 
  The user is actively seeking help, coping mechanisms, definitions, or advice regarding mental health conditions, emotions, symptoms, or psychological well-being. This is a critical class that triggers our clinical knowledge retrieval pipeline.
  (e.g., "How do I deal with panic attacks?", "I'm feeling incredibly anxious right now", "What are the signs of burnout?", "Can you give me tips for depression?")

- out_of_scope: 
  The user is asking about general knowledge, coding, math, recipes, casual chit-chat, or anything completely unrelated to mental health support.
  (e.g., "What is the capital of France?", "Write a python script", "Tell me a joke", "How's the weather?")

Analyze the context carefully. If a user says "Hello, I am having a panic attack", the primary intent is 'asking_mental_health_question', not 'greeting'. Prioritize clinical inquiries over conversational fluff.
"""

In [46]:
class IntentResponse(BaseModel):
    intent: Literal[
        "greeting",
        "goodbye",
        "gratitude",
        "asking_mental_health_question",
        "out_of_scope"
    ] = Field(description="The classified intent of the user's message.")

In [ ]:
class IntentClassifier:

    def __init__(
        self,
        model_name  : str   = "llama-3.3-70b-versatile",  
        temperature : float = 0,
    ):
        logger.info(f"Initializing IntentClassifier with model='{model_name}', temp={temperature}")
        self.model_name  = model_name
        self.temperature = temperature
        self.examples    = self._load_examples()
        self.chain       = self._build_chain()
        logger.info("IntentClassifier initialization complete.")

    def _load_examples(self):
        logger.info("Attempting to load few-shot examples from 'intentExamples.json'...")
        try:
            with open("intentExamples.json", "r", encoding="utf-8") as f:
                examples = json.load(f)
            logger.info(f"Successfully loaded {len(examples)} few-shot examples.")
            return examples
        except Exception as e:
            logger.error(f"Failed to load examples file: {str(e)}")
            raise

    def _build_chain(self):
        logger.info("Assembling LangChain LCEL pipeline components...")

        example_prompt = PromptTemplate(
            input_variables = ["query", "intent"],
            template        = "User: {query}\nIntent: {intent}",
        )

        few_shot_prompt = FewShotPromptTemplate(
            examples        = self.examples,
            example_prompt  = example_prompt,
            prefix          = SYSTEM_PROMPT,
            suffix          = "User: {input}\nIntent:",
            input_variables = ["input"],
        )

        system_message_prompt = SystemMessagePromptTemplate(prompt=few_shot_prompt)
        human_message_prompt  = HumanMessagePromptTemplate.from_template("{input}")
        chat_prompt           = ChatPromptTemplate.from_messages(
            [system_message_prompt, human_message_prompt]
        )
        logger.info("Prompt templates structured successfully.")

        logger.info(f"Connecting to Groq API for model '{self.model_name}'...")  
        llm = ChatGroq(                                   
            model       = self.model_name,
            temperature = self.temperature,
            api_key     = os.getenv("OPENAI_API_KEY"),   
        )

        logger.info("Binding Pydantic output schema (IntentResponse) to LLM...")
        structured_llm = llm.with_structured_output(IntentResponse)

        chain = chat_prompt | structured_llm
        logger.info("LCEL routing chain compiled successfully.")
        return chain

    def predict(self, text: str) -> str:
        logger.info(f"Received user input for prediction: '{text}'")
        logger.info("Invoking LLM chain...")

        try:
            prediction: IntentResponse = self.chain.invoke({"input": text})
            logger.info(f"Extracted intent: '{prediction.intent}'")
            return prediction.intent
        except Exception as e:
            logger.error(f"Error during chain execution: {str(e)}")
            raise

In [74]:
def get_intent_with_context(english_text, history):
    """
    If there is previous history, give the intent classifier
    the last user message as context so it understands follow-ups.
    """
    if history and len(history) >= 2:
        last_user_msg = history[-2]["content"]  # last user turn
        context_input = (
            f"Previous message: {last_user_msg}\n"
            f"Current message: {english_text}"
        )
    else:
        context_input = english_text

    return intent_classifier.predict(context_input)

In [72]:
DIRECT_RESPONSE_PROMPTS = {
    "greeting"   : "The user is greeting you. Respond with a warm, friendly greeting and ask how you can help them today.",
    "goodbye"    : "The user is saying goodbye. Respond with a warm farewell and remind them you are here if they need support.",
    "gratitude"  : "The user is thanking you. Respond with a kind acknowledgment and let them know you are always here to help.",
    "out_of_scope": "The user is asking something outside your scope as a mental health assistant. Politely let them know you can only help with mental health related topics and invite them to ask something relevant.",
}
def get_direct_response(intent, detected_language):
    # No history passed here — these are simple standalone responses
    instruction    = DIRECT_RESPONSE_PROMPTS.get(intent, "Respond helpfully and kindly.")
    system_message = f"""You are a compassionate mental health support assistant.
{instruction}
You MUST respond in {detected_language}.
Keep your response short and natural.
"""
    response = client_llm.chat.completions.create(
        model    = MAIN_MODEL,
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user",   "content": ""},
        ],
        max_tokens  = 256,
        temperature = 0.7,
    )
    return response.choices[0].message.content.strip()

In [50]:
def test_system():
    classifier = IntentClassifier()

    test_cases = {
        "Hey, how's it going?"                        : "greeting",
        "Goodbye, see you tomorrow."                  : "goodbye",
        "Thank you so much for the advice!"           : "gratitude",
        "I'm feeling deeply anxious and can't sleep." : "asking_mental_health_question",
        "Can you write a poem about space?"           : "out_of_scope",
    }

    passed = 0
    for text, expected in test_cases.items():
        result = classifier.predict(text)
        status = "PASS" if result == expected else f"FAIL (got '{result}', expected '{expected}')"
        print(f"{status}  |  '{text}'")
        if result == expected:
            passed += 1

    print(f"\n{passed}/{len(test_cases)} passed")

test_system()

00:46:43 [INFO] Initializing IntentClassifier with model='llama-3.3-70b-versatile', temp=0
00:46:43 [INFO] Attempting to load few-shot examples from 'intentExamples.json'...
00:46:43 [INFO] Successfully loaded 11 few-shot examples.
00:46:43 [INFO] Assembling LangChain LCEL pipeline components...
00:46:43 [INFO] Prompt templates structured successfully.
00:46:43 [INFO] Connecting to Groq API for model 'llama-3.3-70b-versatile'...
00:46:44 [INFO] Binding Pydantic output schema (IntentResponse) to LLM...
00:46:44 [INFO] LCEL routing chain compiled successfully.
00:46:44 [INFO] IntentClassifier initialization complete.
00:46:44 [INFO] Received user input for prediction: 'Hey, how's it going?'
00:46:44 [INFO] Invoking LLM chain...
00:46:44 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
00:46:44 [INFO] Extracted intent: 'greeting'
PASS  |  'Hey, how's it going?'
00:46:44 [INFO] Received user input for prediction: 'Goodbye, see you tomorrow.'
00:46

## Integration

In [51]:
# Module 3 - Intent Classifier
# intentExamples.json must be in the same folder as this notebook
intent_classifier = IntentClassifier()
print("Intent classifier ready")

00:56:01 [INFO] Initializing IntentClassifier with model='llama-3.3-70b-versatile', temp=0
00:56:01 [INFO] Attempting to load few-shot examples from 'intentExamples.json'...
00:56:01 [INFO] Successfully loaded 11 few-shot examples.
00:56:01 [INFO] Assembling LangChain LCEL pipeline components...
00:56:01 [INFO] Prompt templates structured successfully.
00:56:01 [INFO] Connecting to Groq API for model 'llama-3.3-70b-versatile'...
00:56:02 [INFO] Binding Pydantic output schema (IntentResponse) to LLM...
00:56:02 [INFO] LCEL routing chain compiled successfully.
00:56:02 [INFO] IntentClassifier initialization complete.
Intent classifier ready


In [ ]:
def pipeline(user_message, history=[]):

    detected_language = predict_language(user_message)

    if detected_language != "english":
        english_text = translate_to_english(user_message, detected_language)
    else:
        english_text = user_message

    emotion = "joy"

    # ── use context-aware intent detection ──────────────────────────────────
    intent = get_intent_with_context(english_text, history)  

    if intent == "asking_mental_health_question":
        response = get_rag_answer(user_message, detected_language, emotion, history, top_k=5)
    else:
        response = get_direct_response(intent, detected_language)  # ← no history

    history.append({"role": "user",      "content": user_message})
    history.append({"role": "assistant", "content": response})

    return {
        "response" : response,
        "intent"   : intent,
        "emotion"  : emotion,
        "language" : detected_language,
        "history"  : history,
    }

In [60]:
# ── Test 1 : English mental health question → should trigger RAG ────────────
result = pipeline("I have been feeling very anxious and cannot sleep at night")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
# print(f"Emotion  : {result['emotion']}")
print(f"Response : {result['response']}")
print("=" * 60)

01:15:36 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:15:36 [INFO] Received user input for prediction: 'I have been experiencing a great deal of anxiety and am having trouble falling asleep at night.'
01:15:36 [INFO] Invoking LLM chain...
01:15:37 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:15:37 [INFO] Extracted intent: 'asking_mental_health_question'
01:15:37 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.55it/s]


01:15:37 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points/query "HTTP/1.1 200 OK"
01:15:38 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points "HTTP/1.1 200 OK"
01:15:38 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Language : English
Intent   : asking_mental_health_question
Response : I'm so glad you reached out for support. It takes a lot of courage to acknowledge and talk about our struggles. Anxiety can be overwhelming, and it's completely normal to have trouble sleeping when we're feeling anxious.

From what you've shared, it sounds like you're experiencing a mix of anxiety and difficulty sleeping. I want you to know that you're not alone in this. Many people struggle with similar issues.

To help you feel more grounded, I'd like to offer

In [61]:
# ── Test 2 : Arabic mental health question → translate then RAG ─────────────
result = pipeline("أشعر بالحزن الشديد ولا أعرف كيف أتعامل مع مشاعري")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Emotion  : {result['emotion']}")
print(f"Response : {result['response']}")
print("=" * 60)

01:17:32 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:17:32 [INFO] Received user input for prediction: 'I feel a deep sadness and I don't know how to deal with my emotions.'
01:17:32 [INFO] Invoking LLM chain...
01:17:32 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:17:32 [INFO] Extracted intent: 'asking_mental_health_question'
01:17:32 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.99it/s]


01:17:33 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points/query "HTTP/1.1 200 OK"
01:17:33 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points "HTTP/1.1 200 OK"
01:17:34 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Language : Arabic
Intent   : asking_mental_health_question
Emotion  : joy
Response : أنا هنا للاستماع إليك و ναعيدك بالثقة. شعورك بالأسى عميق هو شيء طبيعي، و يمكنك التعامل معه من خلال التحدث مع شخص يمكنك الثقة به، مثل والدك أو مسؤول صحي. كما يمكنك محاولة التأمل والاسترخاء لتحسين حالتك النفسية. أتمنى لك أن تجد المساعدة التي تحتاجها.


In [65]:
# ── Test 3 : Greeting → should skip RAG entirely ────────────────────────────
result = pipeline("Hello, is anyone there?")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Response : {result['response']}")
print("=" * 60)

01:19:41 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:19:41 [INFO] Received user input for prediction: 'Hello, is anyone there?'
01:19:41 [INFO] Invoking LLM chain...
01:19:41 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:19:41 [INFO] Extracted intent: 'greeting'
01:19:42 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Language : English
Intent   : greeting
Response : Hello and welcome. I'm so glad you're here. How can I support you today? What's been going on, and how are you feeling?


In [66]:
# ── Test 4 : Out of scope → should return polite refusal ────────────────────
result = pipeline("What is the capital of France?")
print(f"Language : {result['language']}")
print(f"Intent   : {result['intent']}")
print(f"Response : {result['response']}")
print("=" * 60)

01:20:08 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:20:08 [INFO] Received user input for prediction: 'The capital of France is Paris.'
01:20:08 [INFO] Invoking LLM chain...
01:20:09 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:20:09 [INFO] Extracted intent: 'out_of_scope'
01:20:09 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Language : English
Intent   : out_of_scope
Response : I'm here to support you with any mental health-related concerns you may have. If you'd like to talk about something specific, such as stress, anxiety, or relationships, I'm here to listen and offer guidance. What's on your mind?


In [75]:
# Simulate a multi-turn conversation
history = []

# Turn 1
result  = pipeline("I feel very anxious and cannot sleep", history)
history = result["history"]
print(f"Turn 1 → {result['response']}\n")

# Turn 2 — LLM now knows context from Turn 1
result  = pipeline("What can I do about it?", history)
history = result["history"]
print(f"Turn 2 → {result['response']}\n")

# Turn 3
result  = pipeline("Thank you that really helps", history)
history = result["history"]
print(f"Turn 3 → {result['response']}\n")

01:45:02 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:45:02 [INFO] Received user input for prediction: 'I'm experiencing a great deal of anxiety and am unable to fall asleep.'
01:45:02 [INFO] Invoking LLM chain...
01:45:03 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
01:45:03 [INFO] Extracted intent: 'asking_mental_health_question'
01:45:03 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.88it/s]


01:45:04 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points/query "HTTP/1.1 200 OK"
01:45:04 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points "HTTP/1.1 200 OK"
01:45:05 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Turn 1 → I'm so glad you reached out for support. It's completely normal to feel anxious, and it's great that you're acknowledging it. First, let me assure you that you're not alone in this experience. Many people have struggled with anxiety and managed to find ways to cope.

Considering your question, I'd like to suggest a few things that might help you calm down and fall asleep. Have you tried a daily mindfulness practice, like deep breathing exercises or guided meditation? These can be really helpful in reducing anxiety and prom

Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]


01:45:06 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points/query "HTTP/1.1 200 OK"
01:45:06 [INFO] HTTP Request: POST https://3e303b12-2c26-4461-b08b-eb9a10d7ab85.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/mental_health_chunks/points "HTTP/1.1 200 OK"
01:45:07 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Turn 2 → I'm so glad you're reaching out for support. It's clear that you're struggling with intense and overwhelming emotions, and that's a really brave thing to acknowledge. From what you've shared, it seems like you might be experiencing some intense anger and frustration, and it's affecting your self-care and relationships.

Firstly, I want to acknowledge that you're already taking steps towards seeking help. That's amazing! It takes a lot of courage to admit when we need support.

Considering your experiences, I think it would